# SigAlg's `std` method

In [ ]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `Operators.std` method in SigAlg is a method for computing *standard deviations* of random variables and vectors, both unconditional and conditional versions. The API reference is [here](https://johnmyers-phd.com/sigalg/api/core/#sigalg.core.Operators.std).

## Mathematical definition

Let $X:\Omega \to \mathbb{R}$ be a random variable on a probability space $(\Omega, \mathcal{F},P)$ for which $E(X^2) < \infty$, and let $\mathcal{G}$ be a sub-$\sigma$-algebra of $\mathcal{F}$. The *conditional standard deviation* of $X$ with respect to $\mathcal{G}$ is any $\mathcal{G}$-measurable random variable $\sigma(X \mid \mathcal{G})$ for which

$$
\sigma(X\mid \mathcal{G}) = \sqrt{V(X\mid \mathcal{G})}.
$$

In the case that $\Omega$ is finite (as it always is, in SigAlg), the $\sigma$-algebra $\mathcal{G}$ is determined by its (finitely many) atoms, and the space $L^2(\Omega, \mathcal{G}, P)$ has an orthogonal basis given by the indicator functions of the atoms of $\mathcal{G}$ with nonzero probability. Then we have

$$
\sigma(X\mid \mathcal{G}) = \sum_B \sigma(X|_B) I_B,
$$

where the sum extends over all atoms $B$ of $\mathcal{G}$ with nonzero probability, and where $\sigma(X|_B)$ is the standard deviation of the restricted random variable $X|_B:B\to \mathbb{R}$ on $B$ equipped with the conditional probability measure $P_B$ with $P_B(C) = P(C)/P(B)$ for $C\subset B$.

If $X : \Omega \to \mathbb{R}^d$ is a random vector of dimension $d>1$, with components

$$
X = (X_1,X_2,\ldots,X_d),
$$

then this method returns a `RandomVector` whose component random variables are the conditional standard deviations $\sigma(X_j \mid \mathcal{G})$, for $j=1,2,\ldots,d$.

## API examples


### Unconditional standard deviations

We begin by defining a sample space $\Omega = \{0,1,2,3,4\}$ and a probability measure $P$ on $\Omega$.

In [12]:
from sigalg.core import ProbabilityMeasure, SampleSpace

Omega = SampleSpace().from_sequence(size=5)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.1,
        1: 0.15,
        2: 0.3,
        3: 0.25,
        4: 0.2,
    }
)

Define a random variable $X: \Omega \to \mathbb{R}$ on the sample space $\Omega$ and set its `probability_measure` attribute to $P$ so that all standard deviations will be computed relative to $P$.

In [13]:
from sigalg.core import RandomVariable

X = RandomVariable(domain=Omega).from_dict(
    {
        0: 1,
        1: 1,
        2: -3,
        3: 2,
        4: -3,
    }
)
X.probability_measure = P

All standard deviations in SigAlg are instances of `RandomVariable`. The unconditional standard deviation `std(X)` is thus a constant random variable whose value is the usual standard deviation $\sigma(X) = \sqrt{V(X)}$. The `item` method extracts $\sigma(X)$ from `std(X)`.

In [14]:
from sigalg.core import Operators

std = Operators.std

stdev_rv = std(X)
stdev = std(X).item()

print(stdev_rv, "\n")
print(stdev)

Random variable 'std(X)':
          std(X)
sample          
0       2.277608
1       2.277608
2       2.277608
3       2.277608
4       2.277608 

2.277608394786075


### Conditional standard deviations

Define a $\sigma$-algebra $\mathcal{G}$ on $\Omega$ with atoms $A_0=\{0,1\}$ and $A_1 = \{2,3,4\}$. 

In [15]:
from sigalg.core import SigmaAlgebra

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
    }
)

Compute the conditional standard deviation $\sigma(X\mid \mathcal{G})$. Notice that $X$ is constant on the atom $A_0$ of $\mathcal{G}$, and hence its standard deviation is $0$ on this atom.

In [16]:
print(std(X,G))

Random variable 'std(X|G)':
        std(X|G)
sample          
0       0.000000
1       0.000000
2       2.357023
3       2.357023
4       2.357023


### Testing properties of standard deviations

#### Conditional standard deviations are linear combinations

We noted in the definition that the conditional standard deviation may be expressed as a linear combination of the indicator functions of the atoms of the $\sigma$-algebra. In the next code cell, we test this. Notice that the output matches the output above.

In [17]:
I = RandomVariable.indicator_of

linear_combo = sum([std(X(A)).item() * I(A) for A in G.to_atoms()])

print(linear_combo.with_name("linear_combo"))

Random variable 'linear_combo':
        linear_combo
sample              
0           0.000000
1           0.000000
2           2.357023
3           2.357023
4           2.357023


#### Squaring the standard deviation yields variance

In the next cell, we check the defining equation

$$
\sigma(X\mid \mathcal{G})^2 = V(X\mid \mathcal{G}).
$$

In [18]:
V = Operators.variance

stdev_sq = std(X,G) ** 2
var = V(X,G)

print(stdev_sq, "\n")
print(var)

Random variable '(std(X|G)**2)':
        (std(X|G)**2)
sample               
0            0.000000
1            0.000000
2            5.555556
3            5.555556
4            5.555556 

Random variable 'V(X|G)':
          V(X|G)
sample          
0       0.000000
1       0.000000
2       5.555556
3       5.555556
4       5.555556


#### Complete information yields no deviation

If $X$ is $\mathcal{H}$-measurable, then we must have $\sigma(X \mid \mathcal{H})=0$. We verify this in the next cell:

In [19]:
X = RandomVariable(domain=Omega).from_dict(
    {
        0: 1,
        1: 1,
        2: -3,
        3: 2,
        4: -3,
    }
)
H = SigmaAlgebra(sample_space=Omega, name="H").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 2,
        4: 3,
    }
)

print(std(X, H))

Random variable 'std(X|H)':
        std(X|H)
sample          
0            0.0
1            0.0
2            0.0
3            0.0
4            0.0
